# Trial video inspection

Utilities to jump from a **trial** (its outbound / inbound path) to **where it is in the raw
video**, and to pull the actual clip - without decoding whole files.

The bridge is that trial times and video frame times are the *same* Bonsai/HARP clock: each
video segment's CSV logs a `Seconds` column, one row per frame, in the same seconds as
`outbound_start_time` etc. So locating an event is a nearest-`Seconds` lookup - no conversion.

This notebook uses the **same mouse/session as `locomotion_speed_plots.ipynb`**
(`FL_M01569519`, its first session), and demonstrates the tools by pulling out the session's
**worst-tracked trial** (lowest DLC keypoint confidence).

In [ ]:
# Setup | imports + knobs (same as locomotion_speed_plots).
import pathlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_conduit.qc import (
    qc_datastructure, training_spec, training_session_names, filter_trials,
    slice_pose_for_trial, DEFAULT_CENTROID_POINTS,
    TRAINING_FIRST_MID_LAST_DETAIL, TRAINING_PHASE_LABELS,
)

ROOT = pathlib.Path('/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training')
CENTROID_POINTS = list(DEFAULT_CENTROID_POINTS)   # ('nose','lear','rear','body','tailbase')
VIDEO_SUBDIR = 'UndistortedVideoData'             # the video the figures / DLC use
MOUSE = 'FL_M01569519'                            # same animal as the locomotion notebook

In [ ]:
# Load the first/mid/last training sessions, filter trials, and pick the demo session.
# (Optionally restrict the load to MOUSE for speed - falls back to loading all if the spec
# has no mouseID column.)
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)
spec['phase'] = spec['training_day'].map(TRAINING_PHASE_LABELS)
if 'mouseID' in spec.columns:
    spec = spec[spec['mouseID'] == MOUSE].copy()

datastructure = qc_datastructure(
    root=ROOT, depth=2, level_names=('mouseID', 'day'),
    streams=('events', 'nosepoke', 'soundcard', 'session_settings', 'video', 'dlc'),
    include=training_session_names(spec),
)
result = datastructure.load()
trials_filtered = filter_trials(result['trials'], spec)
position = result['dlc:position']                # Time x keypoints x space
confidence = result['dlc:confidence']            # Time x keypoints (session coord on Time)

SESSION = trials_filtered.loc[trials_filtered['mouseID'] == MOUSE, 'session'].iloc[0]
print(f'MOUSE={MOUSE}  SESSION={SESSION}  |  {len(trials_filtered[trials_filtered.session==SESSION])} trials in session')

## Video utilities

Locator (time → file/frame/timecode) and clip extractor (in-memory frames / saved .mp4). Nothing is decoded beyond the frames you ask for; the locator reads only CSV `Seconds` columns.

In [ ]:
# Locate trial events (outbound / inbound) in the raw video.
# The trial times and the video frame times share ONE clock: each video segment's CSV logs a
# `Seconds` column - one row per frame, in the SAME Bonsai seconds as outbound_start_time etc.
# So locating an event is a nearest-`Seconds` lookup in that CSV; no decoding and no (slow) clip
# extraction. This just tells you which file and which frame/timecode to scrub to by hand.
import pathlib
from dataclasses import dataclass
import numpy as np
import pandas as pd
import imageio.v3 as iio
import matplotlib.pyplot as plt

_VIDLOC_SEG_COLS = {'outbound': ('outbound_start_time', 'outbound_end_time'),
                    'inbound':  ('inbound_start_time',  'inbound_end_time'),
                    'trial':    ('start_time',          'end_time')}


@dataclass
class VideoLocation:
    session: str
    video_file: pathlib.Path
    frame_in_file: int        # 0-based frame index within video_file
    global_frame: int         # index across the session's segments (matches the pose Time index)
    t_in_file_s: float        # seconds from this file's first frame (scrub position)
    timecode: str             # HH:MM:SS.mmm of t_in_file_s
    bonsai_seconds: float     # the matched frame's actual `Seconds`
    requested_seconds: float
    residual_s: float         # |matched - requested|; large => asked outside the recording


def _timecode(s):
    s = max(float(s), 0.0)
    h, r = divmod(s, 3600)
    m, sec = divmod(r, 60)
    return f'{int(h):02d}:{int(m):02d}:{sec:06.3f}'


def video_segments(session, *, root, video_subdir='UndistortedVideoData', _cache={}):
    """Sorted per-segment (avi_path, seconds[], n_frames, global_offset) for a session.

    Reads only each CSV's `Seconds` column (cheap). Pairs each .avi with its CSV by shared stem
    ('VideoData'), else positionally in sorted (chronological) order, requiring equal counts.
    """
    key = (str(root), session, video_subdir)
    if key in _cache:
        return _cache[key]
    dirs = sorted(pathlib.Path(root).glob(f'**/{session}/{video_subdir}'))
    if not dirs:
        raise FileNotFoundError(f'no {video_subdir} folder for session {session!r} under {root}.')
    vdir = dirs[0]
    avis, csvs = sorted(vdir.glob('*.avi')), sorted(vdir.glob('*.csv'))
    if not avis or not csvs:
        raise FileNotFoundError(f'{vdir} is missing .avi or .csv files.')
    if not all(c.stem == a.stem for a, c in zip(avis, csvs)) and len(avis) != len(csvs):
        raise ValueError(f'{vdir}: {len(avis)} avis vs {len(csvs)} csvs - cannot pair.')
    segs, offset = [], 0
    for i, avi in enumerate(avis):
        csv = next((c for c in csvs if c.stem == avi.stem), csvs[i])
        sec = pd.read_csv(csv, usecols=['Seconds'])['Seconds'].to_numpy(float)
        segs.append((avi, sec, len(sec), offset))
        offset += len(sec)
    _cache[key] = segs
    return segs


def locate_time_in_video(session, bonsai_seconds, *, root, video_subdir='UndistortedVideoData'):
    """Nearest video frame to a Bonsai `Seconds` time -> VideoLocation (no decoding)."""
    segs = video_segments(session, root=root, video_subdir=video_subdir)
    T = float(bonsai_seconds)
    best = None
    for avi, sec, n, off in segs:
        j = int(np.argmin(np.abs(sec - T)))
        res = abs(sec[j] - T)
        score = (0 if sec[0] <= T <= sec[-1] else 1, res)   # prefer the segment that spans T
        if best is None or score < best[0]:
            best = (score, avi, sec, off, j, res)
    _, avi, sec, off, j, res = best
    return VideoLocation(session=session, video_file=avi, frame_in_file=j, global_frame=off + j,
                         t_in_file_s=float(sec[j] - sec[0]), timecode=_timecode(sec[j] - sec[0]),
                         bonsai_seconds=float(sec[j]), requested_seconds=T, residual_s=float(res))


def _pose_at(position, confidence, session, t, keypoints, *, session_coord='session',
             time_coord='Time', space_dim='space', keypoint_dim='keypoints'):
    """(x[k], y[k], conf[k] or None, names) for the frame of `session` nearest Bonsai time t."""
    kps = list(keypoints)
    pos = position.sel({keypoint_dim: kps})
    idx = np.flatnonzero(pos[session_coord].values == session)
    if idx.size == 0:
        return None
    j = int(idx[np.argmin(np.abs(pos[time_coord].values[idx] - t))])
    fr = pos.isel({time_coord: j})
    x = np.atleast_1d(fr.sel({space_dim: 'x'}).values.astype(float))
    y = np.atleast_1d(fr.sel({space_dim: 'y'}).values.astype(float))
    c = (np.atleast_1d(confidence.sel({keypoint_dim: kps}).isel({time_coord: j}).values.astype(float))
         if confidence is not None else None)
    return x, y, c, kps


def _step_back_one_frame(loc, session, *, root, video_subdir):
    """Return the VideoLocation one frame earlier than `loc` (same file).

    Used so an inclusive outbound window ends the frame BEFORE the inbound window starts.

    Parameters
    ----------
    loc : VideoLocation
        The location to step back from.
    session : str
        Session id (for the segment lookup).
    root, video_subdir
        As in `locate_time_in_video`.

    Returns
    -------
    VideoLocation
        The preceding frame, or `loc` unchanged if it is already the file's first frame.
    """
    for avi, sec, n, off in video_segments(session, root=root, video_subdir=video_subdir):
        if avi == loc.video_file and loc.frame_in_file > 0:
            j = loc.frame_in_file - 1
            return VideoLocation(session=session, video_file=avi, frame_in_file=j,
                                 global_frame=off + j, t_in_file_s=float(sec[j] - sec[0]),
                                 timecode=_timecode(sec[j] - sec[0]), bonsai_seconds=float(sec[j]),
                                 requested_seconds=loc.requested_seconds, residual_s=loc.residual_s)
    return loc


def locate_trial_segment(trials, session, trial_index, segment='outbound', *, root,
                         video_subdir='UndistortedVideoData', show_frames=True, player='totem',
                         position=None, confidence=None, centroid_points=None,
                         show_keypoints=True, show_time=True,
                         session_column='session', trial_column='trial_index'):
    """Locate a trial's segment window in the video: paste-ready block + optional preview frames.

    Returns (start_loc, end_loc). Nothing is extracted - it reports the file, frame range and
    timecodes to scrub to manually (the preview reads just the two boundary frames).
    """
    a_c, b_c = _VIDLOC_SEG_COLS[segment]
    sel = trials[(trials[session_column] == session) & (trials[trial_column] == trial_index)]
    if sel.empty:
        raise ValueError(f'no trial_index {trial_index} in session {session!r}.')
    row = sel.iloc[0]
    a, b = row[a_c], row[b_c]
    if pd.isna(a) or pd.isna(b):
        raise ValueError(f'{segment} window undefined for this trial (start={a}, end={b}).')
    start = locate_time_in_video(session, a, root=root, video_subdir=video_subdir)
    end = locate_time_in_video(session, b, root=root, video_subdir=video_subdir)
    # outbound_end_time == inbound_start_time (both are 'Target zone triggered'), and each window
    # is INCLUSIVE, so the boundary frame would otherwise be reported in BOTH segments. Step the
    # outbound end back one frame so the last search frame directly PRECEDES the first inbound
    # frame - clipping the two segments then never duplicates a frame.
    if segment == 'outbound' and end.frame_in_file > start.frame_in_file:
        end = _step_back_one_frame(end, session, root=root, video_subdir=video_subdir)

    same = start.video_file == end.video_file
    print(f'{segment} | session {session}  trial {trial_index}')
    print(f'  file : {start.video_file}')
    if not same:
        print(f'  END in a DIFFERENT segment file: {end.video_file}')
    nfr = (end.global_frame - start.global_frame) + 1
    print(f'  frames {start.frame_in_file}-{end.frame_in_file}'
          + ('' if same else f'  (global {start.global_frame}-{end.global_frame})')
          + f'   ({nfr} frames, {end.bonsai_seconds - start.bonsai_seconds:.2f}s)')
    # `player` names a viewer that exists on this machine (ffplay is NOT installed here; totem is).
    # totem takes --seek SECONDS; vlc/mpv use --start-time / --start if you switch player.
    print(f'  seek  {start.timecode} -> {end.timecode}'
          f'    [{player} --seek {start.t_in_file_s:.0f} "{start.video_file}"]')
    for lab, loc in (('start', start), ('end', end)):
        if loc.residual_s > 0.1:
            print(f'  ! {lab}: nearest frame is {loc.residual_s*1000:.0f}ms from the requested '
                  f'time ({loc.requested_seconds:.3f}s) - may be outside the recording.')

    if show_frames:
        kps = list(centroid_points) if centroid_points is not None else (
            list(position['keypoints'].values) if position is not None else [])
        cmap = plt.get_cmap(_KP_CMAP)
        t0 = start.bonsai_seconds
        fig, axes = plt.subplots(1, 2, figsize=(9, 4))
        for j, (ax, (lab, loc)) in enumerate(zip(axes, (('start', start), ('end', end)))):
            try:
                ax.imshow(iio.imread(str(loc.video_file), index=loc.frame_in_file), origin='upper')
            except Exception as exc:
                ax.text(0.5, 0.5, f'frame read failed:\n{exc}', ha='center', va='center', fontsize=7)
            if show_keypoints and position is not None and kps:      # DLC keypoints + centroid
                got = _pose_at(position, confidence, session, loc.bonsai_seconds, kps)
                if got is not None:
                    x, y, c, names = got
                    for i, nm in enumerate(names):
                        sz = 45 * (max(c[i], 0.05) if c is not None else 1.0)
                        ax.scatter(x[i], y[i], s=sz, color=cmap(i % 10), edgecolors='k',
                                   linewidths=0.4, zorder=3, label=nm if j == 0 else None)
                    ax.scatter(np.nanmean(x), np.nanmean(y), s=90, marker='X', color='white',
                               edgecolors='k', linewidths=1.2, zorder=4)
            if show_time:                                            # session-clock time + offset
                rel = loc.bonsai_seconds - t0
                ax.text(0.02, 0.98, f'{_timecode(loc.bonsai_seconds)}\n+{rel:.2f}s into {segment}',
                        transform=ax.transAxes, va='top', ha='left', color='white', fontsize=8,
                        bbox=dict(boxstyle='round', fc='black', alpha=0.5, ec='none'))
            ax.set_title(f'{lab}: frame {loc.frame_in_file}   {loc.timecode}', fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        if show_keypoints and position is not None and kps:
            axes[0].legend(loc='upper right', fontsize=6, framealpha=0.4)
        fig.suptitle(f'{session}   trial {trial_index}   {segment}', y=1.0)
        fig.tight_layout(); plt.show()
    return start, end


# --- Clip extraction (in-memory frames for inline animation; optional file write) -------------
# Perf note (measured, 1620x1260 @20fps mpeg4-SP on the USB drive, 4s clip): ffmpeg FAST-SEEK
# (-ss before -i) reads a segment in ~0.25s; frame-accurate output-seek ~1.6s; naive frame-by-
# frame decode-from-0 is ~15s (never do that). Raw frames are ~6MB each, so keep clips short.
import imageio_ffmpeg as _iff


def _read_range(avi, off_s, dur_s, *, accurate):
    """Read dur_s seconds from avi starting at off_s (seconds from file start) -> (frames, fps)."""
    ip = [] if accurate else ['-ss', f'{off_s:.4f}']          # fast: seek before -i (keyframe jump)
    op = (['-ss', f'{off_s:.4f}'] if accurate else []) + ['-t', f'{max(dur_s, 0):.4f}']
    gen = _iff.read_frames(str(avi), pix_fmt='rgb24', bpp=3, input_params=ip, output_params=op)
    meta = next(gen)
    W, H = meta['size']
    fps = float(meta.get('fps', 20.0))
    frames = [np.frombuffer(b, np.uint8).reshape(H, W, 3) for b in gen]
    gen.close()                      # we stop early; without this imageio SIGKILLs ffmpeg
    return (np.stack(frames) if frames else np.empty((0, H, W, 3), np.uint8)), fps


def clip_segment(session, t_start, t_end, *, root, video_subdir='UndistortedVideoData',
                 accurate=True, pad_s=0.0):
    """Frames (n x H x W x 3) for the Bonsai-time window [t_start, t_end], across segment files.

    accurate=True is frame-exact (~1.6s/4s); False fast-seeks to a keyframe (~0.25s, may start a
    few frames early). pad_s adds context on each side. Returns (frames, fps, files_used).
    """
    segs = video_segments(session, root=root, video_subdir=video_subdir)
    a, b = float(t_start) - pad_s, float(t_end) + pad_s
    parts, used, fps = [], [], 20.0
    for avi, sec, n, off in segs:
        lo, hi = max(a, sec[0]), min(b, sec[-1])
        if hi <= lo:
            continue                                          # window does not touch this segment
        frames, fps = _read_range(avi, lo - sec[0], hi - lo, accurate=accurate)
        if len(frames):
            parts.append(frames); used.append(avi.name)
    if not parts:
        raise ValueError(f'window [{a:.3f},{b:.3f}]s falls outside {session} video coverage.')
    return np.concatenate(parts, 0), fps, used


def animate_trial_segment(trials, session, trial_index, segment='outbound', *, root,
                          video_subdir='UndistortedVideoData', accurate=True, pad_s=0.25,
                          session_column='session', trial_column='trial_index'):
    """Inline HTML animation of a trial's segment clip (no file written). Returns the anim."""
    from matplotlib import animation
    from IPython.display import HTML
    a_c, b_c = _VIDLOC_SEG_COLS[segment]
    sel = trials[(trials[session_column] == session) & (trials[trial_column] == trial_index)]
    if sel.empty:
        raise ValueError(f'no trial_index {trial_index} in session {session!r}.')
    row = sel.iloc[0]
    frames, fps, used = clip_segment(session, row[a_c], row[b_c], root=root,
                                     video_subdir=video_subdir, accurate=accurate, pad_s=pad_s)
    fig, ax = plt.subplots(figsize=(6, 6 * frames.shape[1] / frames.shape[2]))
    im = ax.imshow(frames[0], origin='upper'); ax.set_xticks([]); ax.set_yticks([])
    ttl = ax.set_title('', fontsize=9)

    def _update(k):
        im.set_data(frames[k]); ttl.set_text(f'{session} trial {trial_index} {segment} — frame {k+1}/{len(frames)}')
        return im, ttl
    anim = animation.FuncAnimation(fig, _update, frames=len(frames),
                                   interval=1000 / max(fps, 1), blit=False)
    plt.close(fig)
    print(f'{segment} clip: {len(frames)} frames @ {fps:.1f}fps ({len(frames)/fps:.1f}s), files={used}')
    return HTML(anim.to_jshtml())


def save_trial_clip(trials, session, trial_index, segment, out_path, *, root,
                    video_subdir='UndistortedVideoData', accurate=True, pad_s=0.0,
                    session_column='session', trial_column='trial_index'):
    """Write a trial segment clip to out_path (.mp4). Returns the path."""
    a_c, b_c = _VIDLOC_SEG_COLS[segment]
    row = trials[(trials[session_column] == session) & (trials[trial_column] == trial_index)].iloc[0]
    frames, fps, _ = clip_segment(session, row[a_c], row[b_c], root=root,
                                  video_subdir=video_subdir, accurate=accurate, pad_s=pad_s)
    iio.imwrite(str(out_path), frames, fps=fps, codec='libx264')
    print(f'wrote {out_path}  ({len(frames)} frames, {len(frames)/fps:.1f}s)')
    return out_path


# --- Overlay animation: keypoints + centroid drawn on the frames, saved to a playable mp4 -----
# Since VS Code can't play the raw .avi, we bake the pose overlay into an h264 mp4 and open it in
# an external window (side-by-side with the notebook). Video frames are aligned to the pose EXACTLY
# via accurate ffmpeg seek + fixed frame count (verified pixel-identical to indexed frames).
import subprocess
from matplotlib import animation

_KP_CMAP = 'tab10'


def _aligned_frames(session, win_times, *, root, video_subdir):
    """Read the exact video frames whose Seconds equal `win_times` (per-file, accurate seek)."""
    segs = video_segments(session, root=root, video_subdir=video_subdir)
    assign = []
    for t in win_times:                                     # map each time -> (segment, local frame)
        best = None
        for si, (avi, sec, n, off) in enumerate(segs):
            j = int(np.argmin(np.abs(sec - t)))
            score = (0 if sec[0] <= t <= sec[-1] else 1, abs(sec[j] - t))
            if best is None or score < best[0]:
                best = (score, si, j)
        assign.append((best[1], best[2]))
    frames, i = [], 0
    while i < len(assign):                                  # read each file's contiguous run at once
        si, a = assign[i]
        k = i
        while k + 1 < len(assign) and assign[k + 1] == (si, assign[k][1] + 1):
            k += 1
        avi, sec, n, off = segs[si]
        off0 = float(sec[a] - sec[0])
        gen = _iff.read_frames(str(avi), pix_fmt='rgb24', bpp=3, input_params=[],
                               output_params=['-ss', f'{off0:.6f}', '-frames:v', str(k - i + 1)])
        meta = next(gen)
        W, H = meta['size']
        frames.extend(np.frombuffer(b, np.uint8).reshape(H, W, 3) for b in gen)
        gen.close()                  # bounded read -> close, or imageio SIGKILLs ffmpeg
        i = k + 1
    return np.stack(frames)


def overlay_animation(position, confidence, trials, session, trial_index, segment='outbound', *,
                      root, centroid_points, video_subdir='UndistortedVideoData',
                      keypoint_dim='keypoints', space_dim='space', time_coord='Time',
                      out_path=None, open_window=True, trail=8, dotsize=45, fps=20,
                      show_time=True, show_keypoints=True,
                      session_column='session', trial_column='trial_index'):
    """Render an mp4 of a trial segment with DLC keypoints + centroid overlaid; open it externally.

    Frame k shows the animal with each centroid keypoint (coloured, size ∝ its likelihood) and the
    centroid (white X) at their tracked positions, plus a fading centroid trail. Returns out_path.
    """
    sel = trials[(trials[session_column] == session) & (trials[trial_column] == trial_index)]
    if sel.empty:
        raise ValueError(f'no trial_index {trial_index} in session {session!r}.')
    row = sel.iloc[0]
    kps = list(centroid_points)
    pos = slice_pose_for_trial(position.sel({keypoint_dim: kps}), row, segment=segment,
                               time_coord=time_coord)
    conf = slice_pose_for_trial(confidence.sel({keypoint_dim: kps}), row, segment=segment,
                                time_coord=time_coord)
    if pos.sizes.get(time_coord, 0) < 2:
        raise ValueError(f'{segment} window has too few pose frames.')
    win_times = pos[time_coord].values
    frames = _aligned_frames(session, win_times, root=root, video_subdir=video_subdir)
    n = min(len(frames), pos.sizes[time_coord])
    frames = frames[:n]
    X = pos.sel({space_dim: 'x'}).values[:n]                # (n, k)
    Y = pos.sel({space_dim: 'y'}).values[:n]
    C = conf.values[:n]                                     # (n, k)
    cx = np.nanmean(X, axis=1); cy = np.nanmean(Y, axis=1)  # centroid per frame (display)
    bonsai = win_times[:n]; rel = bonsai - bonsai[0]        # session-clock + offset per frame
    cmap = plt.get_cmap(_KP_CMAP)
    kcols = [cmap(i % 10) for i in range(len(kps))]

    H, W = frames.shape[1:3]
    fig, ax = plt.subplots(figsize=(W / 150, H / 150), dpi=150)
    fig.subplots_adjust(0, 0, 1, 1)
    im = ax.imshow(frames[0], origin='upper')
    scat = [ax.scatter([], [], s=dotsize, color=kcols[i], edgecolors='k', linewidths=0.4,
                       label=kps[i], zorder=3) for i in range(len(kps))]
    cen = ax.scatter([], [], s=90, marker='X', color='white', edgecolors='k', linewidths=1.2, zorder=4)
    trail_ln, = ax.plot([], [], '-', color='yellow', lw=1.2, alpha=0.8, zorder=2)
    txt = ax.text(0.01, 0.99, '', transform=ax.transAxes, va='top', ha='left', color='white',
                  fontsize=8, bbox=dict(boxstyle='round', fc='black', alpha=0.5, ec='none'))
    if show_keypoints:
        ax.legend(loc='upper right', fontsize=6, framealpha=0.4)
    else:
        for s in scat:
            s.set_visible(False)
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_xticks([]); ax.set_yticks([])

    def _update(f):
        im.set_data(frames[f])
        if show_keypoints:
            for i in range(len(kps)):
                scat[i].set_offsets([[X[f, i], Y[f, i]]])
                scat[i].set_sizes([dotsize * max(C[f, i], 0.05)])   # shrink low-confidence dots
        cen.set_offsets([[cx[f], cy[f]]])
        lo = max(0, f - trail)
        trail_ln.set_data(cx[lo:f + 1], cy[lo:f + 1])
        lines = [f'{session}  trial {trial_index}  {segment}']
        if show_time:
            lines.append(f'{_timecode(bonsai[f])}   +{rel[f]:.2f}s')
        lines.append(f'frame {f+1}/{n}   min-conf {np.nanmin(C[f]):.2f}')
        txt.set_text('\n'.join(lines))
        return [im, cen, trail_ln, txt, *scat]

    anim = animation.FuncAnimation(fig, _update, frames=n, interval=1000 / max(fps, 1), blit=False)
    if out_path is None:
        out_path = pathlib.Path(f'/tmp/{session}_trial{trial_index}_{segment}_overlay.mp4')
    out_path = pathlib.Path(out_path)
    plt.rcParams['animation.ffmpeg_path'] = _iff.get_ffmpeg_exe()
    writer = animation.FFMpegWriter(fps=fps, codec='libx264', extra_args=['-pix_fmt', 'yuv420p'])
    anim.save(str(out_path), writer=writer)
    plt.close(fig)
    print(f'wrote overlay: {out_path}  ({n} frames, {n/fps:.1f}s)')
    if open_window:
        try:
            subprocess.Popen(['xdg-open', str(out_path)],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print('opened in the system video player (new window).')
        except Exception as exc:
            print(f'could not auto-open ({exc}); open it manually: {out_path}')
    return out_path

## Find the worst-tracked trial in the session

For each trial we slice the per-keypoint confidence to the trial window and score it by how
often its *worst* keypoint drops below `threshold`. The trial with the highest `frac_below`
is the one whose path is least trustworthy - the natural candidate to eyeball in the video.

In [ ]:
def trial_keypoint_badness(confidence, trials, session, *, centroid_points, threshold=0.6,
                           segment='trial', trial_column='trial_index',
                           session_column='session', time_coord='Time'):
    """Rank a session's trials by DLC keypoint-confidence badness (worst first).

    Per trial: slice the per-keypoint confidence to the segment window, take the per-frame
    WORST keypoint, and summarise -> frac_below (fraction of frames under `threshold`), the
    mean/min of that worst-keypoint trace, and which keypoint is weakest on average.
    """
    kp_conf = confidence.sel(keypoints=list(centroid_points))
    sub = trials[trials[session_column] == session]
    rows = []
    for _, tr in sub.iterrows():
        sl = slice_pose_for_trial(kp_conf, tr, segment=segment, time_coord=time_coord)
        if sl.sizes.get(time_coord, 0) == 0:
            continue
        worst = sl.min('keypoints').values                     # per-frame worst keypoint
        per_kp = sl.mean(time_coord).to_series()               # mean confidence per keypoint
        rows.append(dict(trial_index=int(tr[trial_column]), n_frames=int(len(worst)),
                         frac_below=float(np.mean(worst < threshold)),
                         mean_worst=float(np.nanmean(worst)),
                         min_worst=float(np.nanmin(worst)),
                         weakest_keypoint=str(per_kp.idxmin())))
    return (pd.DataFrame(rows)
            .sort_values('frac_below', ascending=False).reset_index(drop=True))


bad = trial_keypoint_badness(confidence, trials_filtered, SESSION,
                             centroid_points=CENTROID_POINTS, threshold=0.6)
WORST = int(bad.iloc[0]['trial_index'])
print(f'worst-tracked trial in {SESSION}: trial_index={WORST}  '
      f'({bad.iloc[0]["frac_below"]*100:.1f}% of frames below 0.6, '
      f'weakest keypoint = {bad.iloc[0]["weakest_keypoint"]})')
bad.head(10)

## Locate it in the video — with keypoints + time

Where the worst trial's outbound/inbound paths sit in the video. The boundary frames are previewed
inline with the DLC **keypoints + centroid** overlaid and a **time** label (session clock + offset
into the segment). `show_keypoints` / `show_time` toggle each; the same toggles apply to the
overlay animation below.

In [ ]:
# Where the worst trial sits in the video, with DLC keypoints + time drawn on the boundary
# frames. Toggle with show_keypoints / show_time.
for seg in ('outbound', 'inbound'):
    try:
        locate_trial_segment(trials_filtered, SESSION, WORST, seg, root=ROOT,
                             video_subdir=VIDEO_SUBDIR, show_frames=True,
                             position=position, confidence=confidence,
                             centroid_points=CENTROID_POINTS,
                             show_keypoints=True, show_time=True)
    except Exception as exc:
        print(f'{seg}: {exc}')

## Overlay keypoints + centroid on the clip → open in a new window

VS Code can't play the raw `.avi`, so we bake the DLC keypoints and centroid onto the worst
trial's frames and save a **playable h264 mp4**, then open it in the system video player — a
**separate window** to view side-by-side with this notebook. Video frames are aligned to the
pose exactly (accurate ffmpeg seek + fixed frame count, verified pixel-identical).

Each keypoint dot shrinks with its DLC likelihood, so on a badly-tracked trial you *see* which
keypoint the tracker lost. `animate_trial_segment` / `save_trial_clip` remain for a plain
inline animation or a raw (un-overlaid) clip.

In [ ]:
# Overlay the worst trial's outbound path (keypoints + centroid) and open it in a new window.
overlay_animation(position, confidence, trials_filtered, SESSION, WORST, 'outbound',
                  root=ROOT, centroid_points=CENTROID_POINTS, video_subdir=VIDEO_SUBDIR,
                  trail=8, open_window=True)

In [ ]:
# Alternatives:
# - plain inline animation (no overlay), rendered in this cell's output:
# animate_trial_segment(trials_filtered, SESSION, WORST, 'outbound', root=ROOT, video_subdir=VIDEO_SUBDIR)
# - save a raw (un-overlaid) clip to disk:
# save_trial_clip(trials_filtered, SESSION, WORST, 'outbound',
#                 f'/tmp/{SESSION}_trial{WORST}_outbound.mp4', root=ROOT, video_subdir=VIDEO_SUBDIR)

## DeepLabCut API — tools that could improve this

Checked against DLC 3.0 source (GitHub `main`). The two most actionable items were **verified
directly against the source** (`post_processing/filtering.py`, `refine_training_dataset/outlier_frames.py`).

**The catch — DLC's high-level functions need a project, not just an h5.**
`filterpredictions`, `extract_outlier_frames`, `create_labeled_video`, `plot_trajectories` all
take a `config.yaml` as their **first argument** and find the pose by a scorer-named filename
convention (`<video><DLCscorer>.h5`). We consume independently-exported h5s, so none can be
pointed at our files without fabricating a project and renaming the h5s. *(Verified: both
`filterpredictions(config, video, …)` and `extract_outlier_frames(config, videos, …)` take
`config` first.)*

**Three config-free building blocks worth borrowing:**

1. **Spline gap-fill — `deeplabcut.post_processing.filtering.columnwise_spline_interp(data, max_gap=0)`** *(verified: no `config` arg).* Cubic-spline interpolation of NaN gaps up to `max_gap` frames, longer gaps left NaN. A drop-in alternative to our linear `interpolate_over_time` if we want smoother fills — same gap-cap idea as `INTERP_MAX_GAP`, but spline. (No docstring → treat as borrow-the-algorithm / pin a version.)

2. **ARIMA outlier score — `refine_training_dataset.outlier_frames.compute_deviations(Dataframe, dataname, p_bound, alpha, ARdegree, MAdegree)`** + `FitSARIMAXModel(x, p, pcutoff, …)` *(verified: no `config`).* Fits a per-keypoint SARIMAX model (likelihood `< p_bound` treated as missing) and returns a per-frame deviation-from-fit + significance — a smarter "badly-tracked frame" score than our min-likelihood, operating directly on the bodyparts×{x,y,likelihood} schema.

3. **"Jump" detection is trivial — reimplement, don't import.** DLC's `outlieralgorithm='jump'` is just *(verified)*:
   ```python
   temp = df.diff()**2            # squared frame-to-frame displacement
   temp.drop('likelihood', level='coords')
   sum_ = temp.groupby(level='bodyparts').sum()
   flag frames where (sum_ > epsilon**2).any()
   ```
   i.e. squared inter-frame displacement over a threshold — essentially the **neighbour-deviation spike test already built in `locomotion_speed_plots` (Figure 1b-dev)**. Worth aligning the two rather than adding a DLC dependency.

**What DLC does *not* provide (so our code stays):**
- **No bodypart→centroid utility** anywhere in DLC — our centroid (mean of keypoints) is ours to keep (movement has none either).
- **No timestamp/event → video-clip tools** — the locate/clip utilities in this notebook are outside DLC's scope.
- DLC also **never thresholds the h5 itself**: `pcutoff` is applied only at plotting/consumption time — exactly our "filter at use, keep the raw h5" approach.

**Recommendation.** The one low-risk win is **(1) `columnwise_spline_interp`** (or replicating a
capped spline fill) as an interpolation option in the locomotion pipeline. (2)/(3) are worth it
only if min-confidence + neighbour-deviation prove insufficient. Adopting the high-level
renderers (`create_labeled_video`) isn't worth the project scaffolding — our own frame/overlay
plotting is simpler and already config-free.